In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_squared_error

In [3]:
df_main = pd.read_csv(r"D:\Insightrix Project\Real Estate Project\df_main_cleaned.csv") 

In [4]:
# Create a working copy
df_model = df_main.copy()

In [5]:
# Clean and filter required columns
columns_to_use = ['year_built', 'number_of_units', 'sale_price', 'property_city', 'walk_Score', 'transit_Score']
df_model = df_model[columns_to_use].copy()

In [6]:
# Drop rows with missing sale_price
df_model = df_model.dropna(subset=['sale_price'])

In [7]:
# Impute missing values in features using median
df_model['year_built'] = pd.to_numeric(df_model['year_built'], errors='coerce')
df_model['number_of_units'] = pd.to_numeric(df_model['number_of_units'], errors='coerce')
df_model['walk_Score'] = pd.to_numeric(df_model['walk_Score'], errors='coerce')
df_model['transit_Score'] = pd.to_numeric(df_model['transit_Score'], errors='coerce')

df_model = df_model.dropna()

In [8]:
# Encode categorical column 'property_city'
le = LabelEncoder()
df_model['property_city'] = le.fit_transform(df_model['property_city'].astype(str))

In [9]:
# Define features and target
X = df_model.drop(columns='sale_price')
y = df_model['sale_price']



In [10]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [11]:
# Train Random Forest
rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)


RandomForestRegressor(max_depth=10, n_estimators=200, random_state=42)

In [ ]:
# Predict and evaluate
y_pred = rf.predict(X_test)
r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))


In [13]:

# Feature importances
feature_importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)



In [14]:
r2, rmse, feature_importance

(0.5216131662910662,
 1118929.3828805368,
 number_of_units    0.841471
 year_built         0.139974
 property_city      0.018555
 walk_Score         0.000000
 transit_Score      0.000000
 dtype: float64)

# Conclussion

R² Score (0.52) model explains around 52% of the variation in sale prices. That’s okay, but it can definitely be improved with more strong variables (like property size or location quality).

RMSE (~11.19 Lakhs) → On average, the model’s predicted price is off by ~₹11.19 lakhs (or whatever currency the prices are in). This is the typical size of the error

As there is different columns in another dataset and as you mentioned to train model over different five metrices but as you can see that is not dependent or related to each other or doesn't have any impact 

In [ ]:
# Normalize accents and text in property_region and create new column 'market'
import unicodedata

def normalize_text(text):
    if isinstance(text, str):
        text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')
        return text.strip().lower()
    return text

In [16]:
# Apply normalization
df_main['City'] = df_main['property_region'].apply(normalize_text)

# Unique values after normalization
unique_market_values = df_main['City'].dropna().unique().tolist()
unique_market_values

['montreal', 'longueuil', 'laval']

In [20]:
leads_df= pd.read_csv(r"D:\Insightrix Project\Real Estate Project\Leads-Grid 5 (1).csv")

In [21]:
# Apply normalization
leads_df['City'] = leads_df['City'].apply(normalize_text)

# Unique values after normalization
unique_market_values = leads_df['City'].dropna().unique().tolist()
unique_market_values

['montreal']

In [24]:
# Re-define clean_currency since environment may have reset
def clean_currency(x):
    if pd.isna(x): return np.nan
    return float(str(x).replace('$', '').replace(',', '').strip())

# Re-apply conversion
leads_df['sale_price'] = leads_df['Ask_Price'].apply(clean_currency)

# Display first few rows for confirmation
leads_df[['Ask_Price', 'sale_price']].head()


,Ask_Price,sale_price
0,$13875000.00,13875000.0
1,$8950000.00,8950000.0
2,$8950000.00,8950000.0
3,$7995000.00,7995000.0
4,$5595000.00,5595000.0


In [25]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import numpy as np

leads_df['sale_price'] = leads_df['Ask_Price'].apply(clean_currency)
leads_df['number_of_units'] = pd.to_numeric(leads_df['Units'], errors='coerce')
leads_df['year_built'] = pd.to_numeric(leads_df['Year_Built'], errors='coerce')
leads_df['walk_Score'] = pd.to_numeric(leads_df['Walk_Score'], errors='coerce')
leads_df['transit_Score'] = pd.to_numeric(leads_df['Transit_Score'], errors='coerce')
leads_df['City'] = leads_df['City'].apply(normalize_text)



In [26]:
# Drop rows with missing target (Ask_Price)
leads_model = leads_df.dropna(subset=['sale_price'])

In [27]:
# Fill feature NAs with median
features = ['number_of_units', 'year_built', 'walk_Score', 'transit_Score', 'City']
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder


In [28]:

# Encode City
le = LabelEncoder()
leads_model['City'] = le.fit_transform(leads_model['City'].astype(str))



C:\Users\gzar9\AppData\Local\Temp\ipykernel_21304\2512220386.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  leads_model['City'] = le.fit_transform(leads_model['City'].astype(str))


In [29]:
# Define X and y
X = leads_model[features]
y = leads_model['sale_price']



In [30]:
# Impute missing values
imp = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imp.fit_transform(X), columns=X.columns)

# Train Random Forest
rf = RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42)
rf.fit(X_imputed, y)



RandomForestRegressor(max_depth=6, n_estimators=150, random_state=42)

In [31]:
# Predict prices for full dataset (including missing)
leads_df['City'] = leads_df['City'].astype(str).apply(normalize_text)
leads_df['City'] = leads_df['City'].map(lambda x: le.transform([x])[0] if x in le.classes_ else -1)



In [32]:
X_all = leads_df[features]
X_all_imputed = pd.DataFrame(imp.transform(X_all), columns=features)
leads_df['Forecasted_Sale_Price'] = rf.predict(X_all_imputed)



In [33]:
lead_df.columns 

Index(['Address', 'Acquisition_Status', 'Neighbourhood', 'Units', 'Cap_Rate',
       'Forecast_YOC', 'Price/Door', 'Ask_Price', 'Predicted Sale Price',
       'Internal_Note', 'DOM', 'Underwriting_URL', 'Lead_Date', 'City',
       'Contacts', 'Lead_Channel', 'Listing_URL', 'Gross_Revenue',
       'Municipal_Value', 'Building_Area', 'Walk_Score', 'Transit_Score',
       'Year_Built', 'Modelled', 'Under_Offer/Sold', 'Removed_from_Market',
       'Underwriting', 'Acquisition_Net_Revenue', 'Acquisition_NOI',
       'Potential_Project_Cost', 'Forecast_Net_Revenue', 'Forecast_NOI',
       'Underwriting_Date', 'Documents', 'Legal Documents', 'Google_Maps_Link',
       'Created_By', 'Last_Modified_By', 'Tasks', 'Call_Logs', 'Email_Logs',
       'Email_Attachment', 'Priority_Level', 'Offer_Date', 'Closing_Date',
       'Data_Room_URL', 'GR/Price', 'Task_Type', 'Task_Status',
       'Walkscore_URL'],
      dtype='object')

In [35]:
predicted_only = leads_df[leads_df['Forecasted_Sale_Price'].notnull()][
    ['Address', 'City', 'year_built', 'number_of_units', 'walk_Score', 'transit_Score', 'Forecasted_Sale_Price']
]
predicted_only.head(10)

,Address,City,year_built,number_of_units,walk_Score,transit_Score,Forecasted_Sale_Price
0,"710 - 730, 30e Avenue",0,1968.0,61.0,77.0,54.0,9.881581e+06
1,"180, boulevard Saint-Joseph",0,2022.0,18.0,61.0,57.0,8.222905e+06
2,"170, boulevard Saint-Joseph",0,2022.0,18.0,61.0,57.0,8.222905e+06
3,"480Z - 510Z, 2e Avenue",0,1958.0,26.0,72.0,55.0,6.967325e+06
4,"4280, Rue Broadway",0,1958.0,21.0,67.0,49.0,5.346145e+06
5,"805, 25e Avenue",0,1958.0,21.0,64.0,53.0,5.374991e+06
6,"965 - 985, 13e Avenue",0,1957.0,18.0,66.0,51.0,5.013039e+06
7,"724 - 726, 12e Avenue",0,NaN,12.0,83.0,54.0,2.611694e+06
8,"100 - 120, 18e Avenue",0,1959.0,13.0,88.0,49.0,2.698913e+06
9,"105, Rue Milton",0,1964.0,112.0,99.0,87.0,1.900181e+07


In [36]:
# Insert forecasted price into the original leads_df
leads_df["forecasted price using random forest"] = leads_df["Forecasted_Sale_Price"]

In [38]:
# Save to specified path
output_path = r"D:\Insightrix Project\Real Estate Project\Leads-Grid 5 (1).csv"
leads_df.to_csv(output_path, index=False)